# Machine Learning-Based Production Time Estimation for Factory Process Optimization

> **Case Study No. 100 | MLF Sem-5 | FY BTech**

---

## Problem Definition

| Item | Detail |
|---|---|
| **Problem Statement** | A factory wants to estimate the time required for a selected production process |
| **Problem Type** | Supervised Regression (predicting continuous production time) |
| **Target Variable** | `production_time_hrs` -- actual time taken to complete a production batch |
| **Project Title** | *ML-Based Production Time Estimation for Factory Process Optimization* |
| **Business Goal** | Predict production time for better scheduling, resource allocation, and bottleneck detection |

### Approach
We predict `production_time_hrs` using manufacturing features (batch size, machine, workers, complexity, etc.).
A factory can use this to:
- Estimate delivery timelines before production starts
- Identify which factors cause delays
- Optimize resource allocation across shifts and machines

In [ ]:
# ============================================================
# CELL 1: Install & Import All Libraries
# ============================================================
!pip install xgboost -q

# --- Core ---
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing ---
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# --- Models ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
import xgboost as xgb

# --- Evaluation ---
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, mean_absolute_percentage_error,
                              roc_curve, roc_auc_score)

# --- Persistence ---
import joblib
import os

# --- Plot Settings ---
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('[OK] All libraries imported successfully.')

---
## Phase 1: Dataset Loading & Documentation

**Dataset:** Synthetic Factory Production Data
**Records:** ~3,520 | **Features:** 16 | **Generated with realistic manufacturing parameters**

### Why synthetic?
1. **Full control** -- We know the ground truth relationships, can verify model quality
2. **Realistic errors** -- Intentional data quality issues (missing values, outliers, typos, duplicates) for proper EDA/preprocessing practice
3. **Right size** -- 3,500+ rows with rich features, no ethical concerns
4. **Direct relevance** -- Columns map exactly to production time estimation

### Feature Description
| Feature | Type | Description |
|---|---|---|
| `order_id` | Int | Unique order identifier |
| `product_type` | Categorical | Type of product (Widget_A/B/C, Gadget_X/Y) |
| `batch_size` | Int | Number of units in the batch |
| `material_grade` | Categorical | Raw material quality (Standard/Premium/Economy) |
| `machine_id` | Categorical | Machine used (M01-M10) |
| `operator_experience_yrs` | Float | Operator experience in years |
| `setup_time_min` | Float | Machine setup time in minutes |
| `complexity_score` | Int | Product complexity (1-5) |
| `ambient_temp_c` | Float | Ambient temperature (Celsius) |
| `humidity_pct` | Float | Humidity percentage |
| `shift` | Categorical | Shift (Morning/Afternoon/Night) |
| `day_of_week` | Categorical | Day of the week |
| `num_workers` | Int | Workers assigned to the batch |
| `defect_rate_pct` | Float | Historical defect rate (%) |
| `planned_time_hrs` | Float | Originally planned production time |
| **`production_time_hrs`** | **Float** | **TARGET -- actual production time in hours** |

In [ ]:
# ============================================================
# CELL 2: Load Dataset
# ============================================================

df = pd.read_csv('factory_production_data.csv')
print(f'[OK] Dataset loaded successfully.')
print(f'     Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'     Columns: {list(df.columns)}')
print(f'     Target: production_time_hrs')
df.head(10)

In [ ]:
# ============================================================
# CELL 3: Initial Data Exploration
# ============================================================

print('=' * 65)
print('DATASET INFO')
print('=' * 65)
df.info()

print('\n' + '=' * 65)
print('STATISTICAL SUMMARY')
print('=' * 65)
display(df.describe().round(3).T)

print('\n' + '=' * 65)
print('MISSING VALUES')
print('=' * 65)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
display(missing_df[missing_df['Count'] > 0])
print(f'Total missing values: {df.isnull().sum().sum()}')

print('\n' + '=' * 65)
print('DUPLICATE ROWS')
print('=' * 65)
print(f'Duplicate rows found: {df.duplicated().sum()}')

print('\n' + '=' * 65)
print('UNIQUE VALUES PER COLUMN')
print('=' * 65)
for col in df.columns:
    print(f'  {col:.<40} {df[col].nunique()} unique values')

print('\n' + '=' * 65)
print('DATA QUALITY FLAGS')
print('=' * 65)
print(f'  Negative setup_time values:  {(df["setup_time_min"] < 0).sum()}')
print(f'  Humidity > 100%:             {(df["humidity_pct"] > 100).sum()}')
print(f'  Unusual product_type values: {df[~df["product_type"].isin(["Widget_A","Widget_B","Widget_C","Gadget_X","Gadget_Y"])].shape[0]}')
print(f'  Material grade inconsistencies: {df["material_grade"].nunique()} unique (expected 3)')

---
## Phase 2: Exploratory Data Analysis (EDA)

Comprehensive visual exploration to understand distributions, relationships, outliers, and patterns.

In [ ]:
# ============================================================
# EDA PLOT 1: Distribution of Target Variable
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Histogram + KDE
axes[0].hist(df['production_time_hrs'], bins=50, color='#4C72B0', edgecolor='white',
             alpha=0.8, density=True, label='Histogram')
df['production_time_hrs'].plot(kind='kde', ax=axes[0], color='red', linewidth=2, label='KDE')
axes[0].axvline(df['production_time_hrs'].mean(), color='orange', linestyle='--',
                linewidth=2, label=f"Mean: {df['production_time_hrs'].mean():.2f}")
axes[0].axvline(df['production_time_hrs'].median(), color='green', linestyle='--',
                linewidth=2, label=f"Median: {df['production_time_hrs'].median():.2f}")
axes[0].set_title('Distribution of Production Time (Target)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Production Time (hours)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Box plot
bp = axes[1].boxplot(df['production_time_hrs'].dropna(), vert=True, patch_artist=True,
                     boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Box Plot -- Production Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Production Time (hours)')

# Log-scale view
axes[2].hist(np.log1p(df['production_time_hrs']), bins=50, color='#55A868', edgecolor='white', alpha=0.8)
axes[2].set_title('Log-Transformed Production Time', fontsize=14, fontweight='bold')
axes[2].set_xlabel('log(1 + Production Time)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print('Observation:')
print(f'  - Target is RIGHT-SKEWED with extreme outliers')
print(f'  - Range: [{df["production_time_hrs"].min():.2f}, {df["production_time_hrs"].max():.2f}] hours')
print(f'  - Mean ({df["production_time_hrs"].mean():.2f}) >> Median ({df["production_time_hrs"].median():.2f}) confirms right skew')
print(f'  - Log transform makes distribution more normal -- consider for modeling')

In [ ]:
# ============================================================
# EDA PLOT 2: Correlation Heatmap
# ============================================================
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8}, annot_kws={"size": 10})
ax.set_title('Correlation Heatmap -- Numerical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr_matrix['production_time_hrs'].drop('production_time_hrs').abs().sort_values(ascending=False)
print('Top Feature Correlations with production_time_hrs:')
for feat, corr_val in target_corr.items():
    direction = '+' if corr_matrix.loc[feat, 'production_time_hrs'] > 0 else '-'
    bar = '#' * int(corr_val * 30)
    print(f'   {direction} {feat:.<35} {corr_matrix.loc[feat, "production_time_hrs"]:+.3f}  {bar}')

In [ ]:
# ============================================================
# EDA PLOT 3: Box Plots for Outlier Detection
# ============================================================
num_features = ['batch_size', 'operator_experience_yrs', 'setup_time_min',
                'complexity_score', 'ambient_temp_c', 'humidity_pct',
                'num_workers', 'defect_rate_pct', 'production_time_hrs', 'planned_time_hrs']

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
axes = axes.flatten()

colors = sns.color_palette('husl', len(num_features))
for idx, col in enumerate(num_features):
    data_clean = df[col].dropna()
    sns.boxplot(y=data_clean, ax=axes[idx], color=colors[idx], width=0.5)
    axes[idx].set_title(col, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('')
    Q1 = data_clean.quantile(0.25)
    Q3 = data_clean.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((data_clean < Q1 - 1.5*IQR) | (data_clean > Q3 + 1.5*IQR)).sum()
    axes[idx].set_xlabel(f'Outliers: {outliers}', fontsize=9, color='red')

plt.suptitle('Box Plots for Outlier Detection', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Observation:')
print('  - batch_size has extreme outliers (injected values up to 5000)')
print('  - production_time_hrs has extreme outliers (injected values up to 200)')
print('  - setup_time_min has NEGATIVE values (data entry errors)')
print('  - humidity_pct has values > 100% (sensor errors)')

In [ ]:
# ============================================================
# EDA PLOT 4: Categorical Feature Analysis
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Product Type
prod_stats = df.groupby('product_type')['production_time_hrs'].mean().sort_values()
prod_stats.plot(kind='barh', ax=axes[0,0], color='#4C72B0', edgecolor='white')
axes[0,0].set_title('Avg Production Time by Product Type', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Avg Production Time (hrs)')

# 2. Machine
machine_stats = df.groupby('machine_id')['production_time_hrs'].mean().sort_values()
machine_stats.plot(kind='barh', ax=axes[0,1], color='#DD8452', edgecolor='white')
axes[0,1].set_title('Avg Production Time by Machine', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Avg Production Time (hrs)')

# 3. Shift
sns.violinplot(data=df, x='shift', y='production_time_hrs', ax=axes[0,2],
               palette='husl', inner='box', order=['Morning','Afternoon','Night'])
axes[0,2].set_title('Production Time by Shift', fontsize=13, fontweight='bold')
axes[0,2].set_ylim(0, df['production_time_hrs'].quantile(0.95) * 1.1)

# 4. Material Grade (shows inconsistencies!)
mat_counts = df['material_grade'].value_counts()
mat_counts.plot(kind='bar', ax=axes[1,0], color=sns.color_palette('Set2'), edgecolor='white')
axes[1,0].set_title('Material Grade Counts\n(Notice inconsistent casing!)', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Material Grade')
axes[1,0].tick_params(axis='x', rotation=45)

# 5. Complexity vs Time
sns.boxplot(data=df, x='complexity_score', y='production_time_hrs', ax=axes[1,1], palette='RdYlGn_r')
axes[1,1].set_title('Complexity Score vs Production Time', fontsize=13, fontweight='bold')
axes[1,1].set_ylim(0, df['production_time_hrs'].quantile(0.95) * 1.1)

# 6. Day of Week
day_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
avail_days = [d for d in day_order if d in df['day_of_week'].values]
day_stats = df.groupby('day_of_week')['production_time_hrs'].mean().reindex(avail_days)
day_stats.plot(kind='bar', ax=axes[1,2], color='#55A868', edgecolor='white')
axes[1,2].set_title('Avg Production Time by Day', fontsize=13, fontweight='bold')
axes[1,2].set_xlabel('Day')
axes[1,2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('Observation:')
print('  - Gadget_X and Widget_C take longest (higher base production rate)')
print('  - Machine efficiency varies significantly (M01-M10)')
print('  - Night shift is slower (fatigue/reduced staffing)')
print('  - material_grade has 6+ unique values instead of 3 -- needs standardization')
print('  - Higher complexity clearly increases production time')

In [ ]:
# ============================================================
# EDA PLOT 5: Key Feature Scatter Plots
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Batch Size vs Production Time
axes[0,0].scatter(df['batch_size'], df['production_time_hrs'], alpha=0.3, c='#4C72B0', s=20)
axes[0,0].set_title('Batch Size vs Production Time', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Batch Size')
axes[0,0].set_ylabel('Production Time (hrs)')

# 2. Operator Experience vs Production Time
axes[0,1].scatter(df['operator_experience_yrs'], df['production_time_hrs'], alpha=0.3, c='#DD8452', s=20)
axes[0,1].set_title('Operator Experience vs Production Time', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Experience (years)')
axes[0,1].set_ylabel('Production Time (hrs)')

# 3. Setup Time vs Production Time
axes[0,2].scatter(df['setup_time_min'], df['production_time_hrs'], alpha=0.3, c='#55A868', s=20)
axes[0,2].set_title('Setup Time vs Production Time\n(Note negative values!)', fontsize=13, fontweight='bold')
axes[0,2].set_xlabel('Setup Time (min)')
axes[0,2].set_ylabel('Production Time (hrs)')

# 4. Num Workers vs Time
axes[1,0].scatter(df['num_workers'], df['production_time_hrs'], alpha=0.3, c='#C44E52', s=20)
axes[1,0].set_title('Num Workers vs Production Time', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Number of Workers')
axes[1,0].set_ylabel('Production Time (hrs)')

# 5. Planned vs Actual Time
axes[1,1].scatter(df['planned_time_hrs'], df['production_time_hrs'], alpha=0.3, c='#8172B2', s=20)
axes[1,1].plot([0, df['planned_time_hrs'].max()], [0, df['planned_time_hrs'].max()],
               'r--', lw=2, label='Perfect Estimate')
axes[1,1].set_title('Planned vs Actual Production Time', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Planned Time (hrs)')
axes[1,1].set_ylabel('Actual Time (hrs)')
axes[1,1].legend()

# 6. Defect Rate vs Time
axes[1,2].scatter(df['defect_rate_pct'], df['production_time_hrs'], alpha=0.3, c='#CCB974', s=20)
axes[1,2].set_title('Defect Rate vs Production Time', fontsize=13, fontweight='bold')
axes[1,2].set_xlabel('Defect Rate (%)')
axes[1,2].set_ylabel('Production Time (hrs)')

plt.tight_layout()
plt.show()

print('Observation:')
print('  - batch_size is the STRONGEST predictor -- clear positive relationship')
print('  - More workers = shorter time (but diminishing returns)')
print('  - Higher experience = slightly less time')
print('  - Planned vs Actual shows most estimates are in the right ballpark')

### EDA -- Key Findings Summary

| # | Finding | Action Required |
|---|---|---|
| 1 | Target is right-skewed with extreme outliers (up to 200 hrs) | Cap outliers using IQR method |
| 2 | `operator_experience_yrs` ~6% missing, `humidity_pct` ~8% missing | Impute with median |
| 3 | `setup_time_min` has 8 negative values | Replace negatives with median |
| 4 | `humidity_pct` has 6 values > 100% | Cap at 100 |
| 5 | `material_grade` has mixed casing (6 variants instead of 3) | Standardize to title case |
| 6 | `product_type` has 5 typos | Fix/map to nearest correct name |
| 7 | 20 duplicate rows | Drop duplicates |
| 8 | `batch_size` is strongest predictor | Must keep |
| 9 | `complexity_score`, `num_workers`, `shift` are significant | Keep and encode |
| 10 | Relationships are non-linear | Tree-based models should outperform linear |

---
## Phase 3: Data Cleaning

Addressing all data quality issues identified during EDA. Each decision is documented with justification.

In [ ]:
# ============================================================
# CELL 4: Data Cleaning
# ============================================================
df_clean = df.copy()
print(f'Shape BEFORE cleaning: {df_clean.shape}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
print()

# 1. Remove Duplicates
dupes = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f'[Step 1] Removed {dupes} duplicate rows')

# 2. Fix product_type typos
typo_map = {'Wiget_A': 'Widget_A', 'Gadgt_X': 'Gadget_X', 'Widget_c': 'Widget_C'}
df_clean['product_type'] = df_clean['product_type'].replace(typo_map)
valid_products = ['Widget_A', 'Widget_B', 'Widget_C', 'Gadget_X', 'Gadget_Y']
bad_products = df_clean[~df_clean['product_type'].isin(valid_products)].shape[0]
print(f'[Step 2] Fixed product_type typos ({bad_products} remaining invalid)')

# 3. Standardize material_grade casing
df_clean['material_grade'] = df_clean['material_grade'].str.strip().str.title()
print(f'[Step 3] Standardized material_grade: {sorted(df_clean["material_grade"].unique())}')

# 4. Fix negative setup_time (replace with median)
neg_setup = (df_clean['setup_time_min'] < 0).sum()
median_setup = df_clean.loc[df_clean['setup_time_min'] >= 0, 'setup_time_min'].median()
df_clean.loc[df_clean['setup_time_min'] < 0, 'setup_time_min'] = median_setup
print(f'[Step 4] Fixed {neg_setup} negative setup_time values -> median ({median_setup:.1f})')

# 5. Cap humidity at 100
bad_humidity = (df_clean['humidity_pct'] > 100).sum()
df_clean.loc[df_clean['humidity_pct'] > 100, 'humidity_pct'] = 100.0
print(f'[Step 5] Capped {bad_humidity} humidity values to 100%')

# 6. Impute missing values with median
for col in ['operator_experience_yrs', 'humidity_pct', 'defect_rate_pct']:
    n_missing = df_clean[col].isnull().sum()
    med_val = df_clean[col].median()
    df_clean[col].fillna(med_val, inplace=True)
    print(f'[Step 6] Imputed {n_missing} missing {col} with median ({med_val:.1f})')

# 7. Cap outliers using IQR method
def cap_outliers(data, column, factor=1.5):
    Q1, Q3 = data[column].quantile(0.25), data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - factor * IQR)
    upper = Q3 + factor * IQR
    n_out = ((data[column] < lower) | (data[column] > upper)).sum()
    data[column] = data[column].clip(lower=lower, upper=upper)
    return n_out, lower, upper

print(f'\n[Step 7] Capping outliers with IQR method:')
for col in ['batch_size', 'production_time_hrs', 'defect_rate_pct', 'planned_time_hrs']:
    n, lo, hi = cap_outliers(df_clean, col)
    print(f'  {col:.<35} {n:>4} outliers capped -> [{lo:.1f}, {hi:.1f}]')

# 8. Drop order_id (not useful for prediction)
df_clean.drop(columns=['order_id'], inplace=True, errors='ignore')
print(f'\n[Step 8] Dropped order_id column (not predictive)')

print(f'\n{"="*60}')
print(f'Shape AFTER cleaning: {df_clean.shape}')
print(f'Remaining missing values: {df_clean.isnull().sum().sum()}')
print(f'Target range: [{df_clean["production_time_hrs"].min():.2f}, {df_clean["production_time_hrs"].max():.2f}]')
print(f'{"="*60}')

---
## Phase 4: Preprocessing & Feature Engineering

Encoding categorical variables, creating new features, and scaling numerical features.

In [ ]:
# ============================================================
# CELL 5: Preprocessing & Feature Engineering
# ============================================================
df_processed = df_clean.copy()

print('FEATURE ENGINEERING')
print('=' * 50)

# 1. Encode product_type
le_product = LabelEncoder()
df_processed['product_type_encoded'] = le_product.fit_transform(df_processed['product_type'])
print(f'[1] Encoded product_type: {dict(zip(le_product.classes_, le_product.transform(le_product.classes_)))}')

# 2. Encode machine_id
le_machine = LabelEncoder()
df_processed['machine_id_encoded'] = le_machine.fit_transform(df_processed['machine_id'])
print(f'[2] Encoded machine_id')

# 3. Encode shift
le_shift = LabelEncoder()
df_processed['shift_encoded'] = le_shift.fit_transform(df_processed['shift'])
print(f'[3] Encoded shift: {dict(zip(le_shift.classes_, le_shift.transform(le_shift.classes_)))}')

# 4. Encode day_of_week
le_day = LabelEncoder()
df_processed['day_encoded'] = le_day.fit_transform(df_processed['day_of_week'])
print(f'[4] Encoded day_of_week')

# 5. One-hot encode material_grade
df_processed = pd.get_dummies(df_processed, columns=['material_grade'], drop_first=True, dtype=int)
print(f'[5] One-hot encoded material_grade')

# 6. Feature engineering
df_processed['batch_per_worker'] = df_processed['batch_size'] / df_processed['num_workers'].replace(0, 1)
df_processed['setup_per_batch'] = df_processed['setup_time_min'] / df_processed['batch_size'].replace(0, 1)
df_processed['time_deviation'] = df_processed['planned_time_hrs'] - df_processed['production_time_hrs']
print(f'[6] Created: batch_per_worker, setup_per_batch, time_deviation')

# 7. Drop original categorical columns
drop_cols = ['product_type', 'machine_id', 'shift', 'day_of_week']
df_processed.drop(columns=drop_cols, inplace=True)
print(f'[7] Dropped original categorical columns: {drop_cols}')

print(f'\n{"="*60}')
print(f'Final processed shape: {df_processed.shape}')
print(f'\nFeatures ({df_processed.shape[1]} columns):')
for i, col in enumerate(df_processed.columns, 1):
    dtype = df_processed[col].dtype
    print(f'   {i:>2}. {col:.<40} {str(dtype):>10}')
print(f'{"="*60}')

df_processed.head()

In [ ]:
# ============================================================
# Visualize Data After Preprocessing
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Feature correlations with target
corr = df_processed.corr(numeric_only=True)
target_corr = corr['production_time_hrs'].drop('production_time_hrs').sort_values()
colors_corr = ['#DD8452' if v < 0 else '#4C72B0' for v in target_corr.values]
target_corr.plot(kind='barh', ax=axes[0], color=colors_corr, edgecolor='white')
axes[0].set_title('Feature Correlations with Target\n(After Preprocessing)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# 2. batch_per_worker distribution
axes[1].hist(df_processed['batch_per_worker'], bins=30, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution of Batch Per Worker\n(Engineered Feature)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Batch Per Worker')
axes[1].set_ylabel('Frequency')

# 3. Target after cleaning
axes[2].hist(df_processed['production_time_hrs'], bins=30, color='#55A868', edgecolor='white', alpha=0.8)
axes[2].axvline(df_processed['production_time_hrs'].mean(), color='red',
                linestyle='--', linewidth=2, label=f"Mean: {df_processed['production_time_hrs'].mean():.2f}")
axes[2].set_title('Target Distribution After Cleaning', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Production Time (hrs)')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.suptitle('Data After Preprocessing', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6: Train-Test Split & Scaling
# ============================================================

# Remove time_deviation (leaks target info) and planned_time_hrs from features
X = df_processed.drop(['production_time_hrs', 'time_deviation'], axis=1)
y = df_processed['production_time_hrs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X.columns, index=X_test.index
)

print('[OK] Train-Test Split & Scaling Complete.')
print(f'{"="*50}')
print(f'   Training set:  {X_train_scaled.shape[0]} samples ({X_train_scaled.shape[0]/len(X)*100:.0f}%)')
print(f'   Testing set:   {X_test_scaled.shape[0]} samples ({X_test_scaled.shape[0]/len(X)*100:.0f}%)')
print(f'   Features:      {X_train_scaled.shape[1]}')
print(f'{"="*50}')
print(f'   Target (train) -- Mean: {y_train.mean():.4f} | Std: {y_train.std():.4f}')
print(f'   Target (test)  -- Mean: {y_test.mean():.4f} | Std: {y_test.std():.4f}')
print(f'   No data leakage: scaler fit on train only')

---
## Phase 5: Model Training

Training **7 regression models** with 5-fold cross-validation:

| # | Model | Type | Why? |
|---|---|---|---|
| 1 | Linear Regression | Baseline | Simple, interpretable benchmark |
| 2 | Ridge Regression | Regularized Linear | Handles multicollinearity |
| 3 | Lasso Regression | Regularized Linear | Feature selection via L1 penalty |
| 4 | Decision Tree | Non-linear | Captures interactions; interpretable |
| 5 | Random Forest | Ensemble | Reduces overfitting; robust |
| 6 | XGBoost | Gradient Boosting | State-of-the-art for tabular data |
| 7 | SVR | Kernel-based | Good with moderate-size datasets |

In [ ]:
# ============================================================
# CELL 7: Train All Models with Cross-Validation
# ============================================================

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=12),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0),
    'SVR': SVR(kernel='rbf', C=10.0)
}

# Models that need scaling
scale_models = {'Linear Regression', 'Ridge Regression', 'Lasso Regression', 'SVR'}

results = {}

print('Training 7 Regression Models with 5-Fold Cross-Validation...')
print()
header = f'{"Model":<22} {"Train R2":>10} {"Test R2":>10} {"MAE":>10} {"RMSE":>10} {"CV R2 (5-fold)":>18}'
print(header)
print('-' * len(header))

for name, model in models.items():
    if name in scale_models:
        Xtr, Xte = X_train_scaled, X_test_scaled
    else:
        Xtr, Xte = X_train, X_test

    model.fit(Xtr, y_train)
    y_pred_train = model.predict(Xtr)
    y_pred_test = model.predict(Xte)
    cv_scores = cross_val_score(model, Xtr, y_train, cv=5, scoring='r2')

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    results[name] = {
        'model': model, 'train_r2': train_r2, 'test_r2': test_r2,
        'mae': mae, 'rmse': rmse, 'cv_mean': cv_mean,
        'cv_std': cv_std, 'y_pred': y_pred_test
    }

    print(f'{name:<22} {train_r2:>10.4f} {test_r2:>10.4f} {mae:>10.4f} {rmse:>10.4f} {cv_mean:>10.4f} +/- {cv_std:.4f}')

print('=' * len(header))
best_name = max(results, key=lambda k: results[k]['test_r2'])
print(f'\n[BEST] Best Model (by Test R2): {best_name} -- R2 = {results[best_name]["test_r2"]:.4f}')

---
## Phase 6: Evaluation & Model Comparison

### Evaluation Metrics Used

| Metric | Why? |
|---|---|
| **MAE** | Interpretable in original units (hours); robust to outliers |
| **RMSE** | Penalizes large errors more heavily |
| **R2 Score** | Proportion of variance explained (0-1) |
| **MAPE** | Business-friendly percentage metric |
| **CV R2** | Robustness check against overfitting |

In [ ]:
# ============================================================
# CELL 8: Model Comparison Table
# ============================================================

comparison_data = []
for name, r in results.items():
    mape = mean_absolute_percentage_error(y_test, r['y_pred'])
    comparison_data.append({
        'Model': name, 'Train R2': r['train_r2'], 'Test R2': r['test_r2'],
        'MAE': r['mae'], 'RMSE': r['rmse'], 'MAPE': mape,
        'CV R2 Mean': r['cv_mean'], 'CV R2 Std': r['cv_std'],
        'Overfit Gap': r['train_r2'] - r['test_r2']
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Test R2', ascending=False).reset_index(drop=True)

print('COMPLETE MODEL COMPARISON TABLE')
print('=' * 100)
display(comparison_df.style
    .highlight_max(subset=['Test R2', 'CV R2 Mean'], color='#90EE90')
    .highlight_min(subset=['MAE', 'RMSE', 'MAPE', 'Overfit Gap'], color='#90EE90')
    .format(precision=4)
    .set_caption('Model Performance Comparison -- Sorted by Test R2'))

best_model_name = comparison_df.iloc[0]['Model']
print(f'\n[BEST] Best Model: {best_model_name}')
print(f'   Test R2:  {comparison_df.iloc[0]["Test R2"]:.4f}')
print(f'   MAE:      {comparison_df.iloc[0]["MAE"]:.4f} hours')
print(f'   RMSE:     {comparison_df.iloc[0]["RMSE"]:.4f} hours')

In [ ]:
# ============================================================
# CELL 9: Model Comparison Visualizations
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

model_names = list(results.keys())
n_models = len(model_names)
colors = sns.color_palette('husl', n_models)

# 1. R2: Train vs Test
x = np.arange(n_models)
width = 0.35
r2_train = [results[m]['train_r2'] for m in model_names]
r2_test = [results[m]['test_r2'] for m in model_names]
axes[0,0].bar(x - width/2, r2_train, width, label='Train R2', color='#4C72B0', alpha=0.85, edgecolor='white')
axes[0,0].bar(x + width/2, r2_test, width, label='Test R2', color='#55A868', alpha=0.85, edgecolor='white')
axes[0,0].set_title('R2 Score: Train vs Test', fontsize=14, fontweight='bold')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(model_names, rotation=45, ha='right', fontsize=9)
axes[0,0].legend()
axes[0,0].set_ylabel('R2 Score')

# 2. MAE
mae_vals = [results[m]['mae'] for m in model_names]
bars = axes[0,1].barh(model_names, mae_vals, color=colors, edgecolor='white', height=0.6)
axes[0,1].set_title('Mean Absolute Error (MAE) -- Lower is Better', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('MAE (hours)')
for bar, val in zip(bars, mae_vals):
    axes[0,1].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

# 3. RMSE
rmse_vals = [results[m]['rmse'] for m in model_names]
bars = axes[1,0].barh(model_names, rmse_vals, color=colors, edgecolor='white', height=0.6)
axes[1,0].set_title('RMSE -- Lower is Better', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('RMSE (hours)')
for bar, val in zip(bars, rmse_vals):
    axes[1,0].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

# 4. CV R2 with error bars
cv_means = [results[m]['cv_mean'] for m in model_names]
cv_stds = [results[m]['cv_std'] for m in model_names]
bars = axes[1,1].barh(model_names, cv_means, xerr=cv_stds, color=colors, edgecolor='white', height=0.6, capsize=5)
axes[1,1].set_title('Cross-Validation R2 (5-Fold) +/- Std Dev', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('CV R2 Score')

plt.suptitle('Model Performance Comparison Dashboard', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 10: Error Analysis -- Best Model
# ============================================================
from scipy import stats

best_result = results[best_model_name]
y_pred_best = best_result['y_pred']
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Actual vs Predicted
axes[0,0].scatter(y_test, y_pred_best, alpha=0.5, c='#4C72B0', edgecolors='white', s=50)
min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
axes[0,0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0,0].fill_between([min_val, max_val], [min_val-1, max_val-1],
                         [min_val+1, max_val+1], alpha=0.1, color='green', label='+/- 1 hr band')
axes[0,0].set_title(f'Actual vs Predicted -- {best_model_name}', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Actual Production Time (hrs)')
axes[0,0].set_ylabel('Predicted Production Time (hrs)')
axes[0,0].legend()

# 2. Residual Distribution
axes[0,1].hist(residuals, bins=30, color='#DD8452', edgecolor='white', alpha=0.8, density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[0,1].plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
                'r-', linewidth=2, label='Normal Fit')
axes[0,1].axvline(x=0, color='black', linestyle='--', lw=1.5)
axes[0,1].set_title('Residual Distribution', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Residual (Actual - Predicted)')
axes[0,1].legend()

# 3. Residuals vs Predicted
axes[1,0].scatter(y_pred_best, residuals, alpha=0.5, c='#55A868', edgecolors='white', s=50)
axes[1,0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1,0].set_title('Residuals vs Predicted Values', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Predicted (hrs)')
axes[1,0].set_ylabel('Residual')

# 4. Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot of Residuals', fontsize=14, fontweight='bold')

plt.suptitle(f'Error Analysis -- {best_model_name}', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Error Summary -- {best_model_name}')
print(f'{"="*50}')
print(f'   Mean Residual:    {residuals.mean():.4f} (should be ~0)')
print(f'   Std of Residuals: {residuals.std():.4f}')
print(f'   % within +/-1hr: {(np.abs(residuals) <= 1.0).mean()*100:.1f}%')
print(f'   % within +/-2hr: {(np.abs(residuals) <= 2.0).mean()*100:.1f}%')

In [ ]:
# ============================================================
# CELL 11: Feature Importance Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

rf_model = results['Random Forest']['model']
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)
rf_importance.plot(kind='barh', ax=axes[0], color='#4C72B0', edgecolor='white')
axes[0].set_title('Random Forest -- Feature Importance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance (Gini)')

xgb_model = results['XGBoost']['model']
xgb_importance = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=True)
xgb_importance.plot(kind='barh', ax=axes[1], color='#DD8452', edgecolor='white')
axes[1].set_title('XGBoost -- Feature Importance', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance (Gain)')

plt.suptitle('Feature Importance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Top 5 Important Features:')
print('\n   Random Forest:')
for feat, imp in rf_importance.tail(5).iloc[::-1].items():
    bar = '#' * int(imp * 50)
    print(f'   {bar:.<25} {feat} ({imp:.4f})')
print('\n   XGBoost:')
for feat, imp in xgb_importance.tail(5).iloc[::-1].items():
    bar = '#' * int(imp * 50)
    print(f'   {bar:.<25} {feat} ({imp:.4f})')

---
### ROC AUC Analysis -- Binary Classification View

ROC AUC is a classification metric. Since this is a regression task, we create a **binary classification view**:
- Convert the problem to: **"Will production finish on time?"**
- `on_time = 1` if `actual_time <= planned_time`, else `0`
- Use regression predictions as scores and compute ROC curves

In [ ]:
# ============================================================
# CELL 12: ROC AUC Curves -- Binary Classification View
# ============================================================
# Convert regression to binary: "meets deadline?"
# We use the median production time as threshold
threshold = y_test.median()
y_test_binary = (y_test <= threshold).astype(int)  # 1 = fast (on time), 0 = slow

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot ROC for each model
for name, r in results.items():
    # Lower predicted time = higher probability of being "on time"
    # So we negate predictions to use as scores
    scores = -r['y_pred']
    try:
        fpr, tpr, _ = roc_curve(y_test_binary, scores)
        auc_val = roc_auc_score(y_test_binary, scores)
        axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', linewidth=2)
    except:
        pass

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.500)')
axes[0].set_title('ROC Curves -- "On Time Prediction"\n(production_time <= median)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(True, alpha=0.3)

# Bar chart of AUC scores
auc_scores = {}
for name, r in results.items():
    scores = -r['y_pred']
    try:
        auc_scores[name] = roc_auc_score(y_test_binary, scores)
    except:
        auc_scores[name] = 0

auc_df = pd.Series(auc_scores).sort_values()
auc_colors = ['#DD8452' if v < 0.8 else '#55A868' for v in auc_df.values]
auc_df.plot(kind='barh', ax=axes[1], color=auc_colors, edgecolor='white')
axes[1].set_title('ROC AUC Scores by Model', fontsize=14, fontweight='bold')
axes[1].set_xlabel('AUC Score')
axes[1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
for i, (name, val) in enumerate(auc_df.items()):
    axes[1].text(val + 0.005, i, f'{val:.3f}', va='center', fontweight='bold', fontsize=10)
axes[1].legend()

plt.tight_layout()
plt.show()

print('ROC AUC Summary (Binary: On-Time vs Late):')
for name, auc in sorted(auc_scores.items(), key=lambda x: -x[1]):
    status = '[GOOD]' if auc >= 0.85 else '[OK]' if auc >= 0.75 else '[WEAK]'
    print(f'   {status} {name:.<35} AUC = {auc:.4f}')

In [ ]:
# ============================================================
# CELL 13: Hyperparameter Tuning (RandomizedSearchCV -- FAST)
# ============================================================
from scipy.stats import randint, uniform

print('Hyperparameter Tuning with RandomizedSearchCV (n_iter=20, 5-fold)...')
print('   This should take about 1-2 minutes.')
print()

# --- Store pre-tuning scores for comparison ---
pre_tune_scores = {
    'Random Forest': results['Random Forest']['test_r2'],
    'XGBoost': results['XGBoost']['test_r2'],
}

# 1. Random Forest
rf_params = {
    'n_estimators': randint(100, 400),
    'max_depth': [8, 10, 15, 20, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 8),
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params, n_iter=20, cv=5, scoring='r2', n_jobs=-1, random_state=42, verbose=0
)
rf_search.fit(X_train, y_train)
rf_tuned_pred = rf_search.predict(X_test)
rf_tuned_r2 = r2_score(y_test, rf_tuned_pred)

print(f'[OK] Random Forest Tuning Complete!')
print(f'   Best Params: {rf_search.best_params_}')
print(f'   Best CV R2:  {rf_search.best_score_:.4f}')
print(f'   Test R2:     {rf_tuned_r2:.4f} (was {pre_tune_scores["Random Forest"]:.4f})')

# 2. XGBoost
xgb_params = {
    'n_estimators': randint(100, 400),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.7, 0.3),
    'colsample_bytree': uniform(0.7, 0.3),
}

xgb_search = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0),
    xgb_params, n_iter=20, cv=5, scoring='r2', n_jobs=-1, random_state=42, verbose=0
)
xgb_search.fit(X_train, y_train)
xgb_tuned_pred = xgb_search.predict(X_test)
xgb_tuned_r2 = r2_score(y_test, xgb_tuned_pred)

print(f'\n[OK] XGBoost Tuning Complete!')
print(f'   Best Params: {xgb_search.best_params_}')
print(f'   Best CV R2:  {xgb_search.best_score_:.4f}')
print(f'   Test R2:     {xgb_tuned_r2:.4f} (was {pre_tune_scores["XGBoost"]:.4f})')

# Pick final best
if rf_tuned_r2 >= xgb_tuned_r2:
    final_best_model = rf_search.best_estimator_
    final_best_name = 'Random Forest (Tuned)'
    final_best_score = rf_tuned_r2
    final_best_pred = rf_tuned_pred
else:
    final_best_model = xgb_search.best_estimator_
    final_best_name = 'XGBoost (Tuned)'
    final_best_score = xgb_tuned_r2
    final_best_pred = xgb_tuned_pred

print(f'\n[BEST] FINAL BEST MODEL: {final_best_name}')
print(f'   R2   = {final_best_score:.4f}')
print(f'   MAE  = {mean_absolute_error(y_test, final_best_pred):.4f}')
print(f'   RMSE = {np.sqrt(mean_squared_error(y_test, final_best_pred)):.4f}')

In [ ]:
# ============================================================
# CELL 14: Before vs After Tuning Comparison
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Build comparison data ---
tuning_comparison = {
    'Random Forest': {
        'Before R2': pre_tune_scores['Random Forest'],
        'After R2': rf_tuned_r2,
        'Before MAE': results['Random Forest']['mae'],
        'After MAE': mean_absolute_error(y_test, rf_tuned_pred),
        'Before RMSE': results['Random Forest']['rmse'],
        'After RMSE': np.sqrt(mean_squared_error(y_test, rf_tuned_pred)),
    },
    'XGBoost': {
        'Before R2': pre_tune_scores['XGBoost'],
        'After R2': xgb_tuned_r2,
        'Before MAE': results['XGBoost']['mae'],
        'After MAE': mean_absolute_error(y_test, xgb_tuned_pred),
        'Before RMSE': results['XGBoost']['rmse'],
        'After RMSE': np.sqrt(mean_squared_error(y_test, xgb_tuned_pred)),
    }
}

tuning_df = pd.DataFrame(tuning_comparison).T
display(tuning_df.style.format(precision=4).set_caption('Before vs After Hyperparameter Tuning'))

# 1. R2 Comparison
x_pos = np.arange(2)
width = 0.3
axes[0].bar(x_pos - width/2, [tuning_comparison['Random Forest']['Before R2'],
            tuning_comparison['XGBoost']['Before R2']], width,
            label='Before Tuning', color='#DD8452', edgecolor='white')
axes[0].bar(x_pos + width/2, [tuning_comparison['Random Forest']['After R2'],
            tuning_comparison['XGBoost']['After R2']], width,
            label='After Tuning', color='#55A868', edgecolor='white')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(['Random Forest', 'XGBoost'])
axes[0].set_title('R2 Score: Before vs After Tuning', fontsize=14, fontweight='bold')
axes[0].set_ylabel('R2 Score')
axes[0].legend()

# 2. MAE Comparison
axes[1].bar(x_pos - width/2, [tuning_comparison['Random Forest']['Before MAE'],
            tuning_comparison['XGBoost']['Before MAE']], width,
            label='Before Tuning', color='#DD8452', edgecolor='white')
axes[1].bar(x_pos + width/2, [tuning_comparison['Random Forest']['After MAE'],
            tuning_comparison['XGBoost']['After MAE']], width,
            label='After Tuning', color='#55A868', edgecolor='white')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['Random Forest', 'XGBoost'])
axes[1].set_title('MAE: Before vs After Tuning', fontsize=14, fontweight='bold')
axes[1].set_ylabel('MAE (hours)')
axes[1].legend()

# 3. All models R2 comparison (including tuned)
all_r2 = {name: r['test_r2'] for name, r in results.items()}
all_r2['RF (Tuned)'] = rf_tuned_r2
all_r2['XGB (Tuned)'] = xgb_tuned_r2
r2_series = pd.Series(all_r2).sort_values()
bar_colors = ['#55A868' if 'Tuned' in name else '#4C72B0' for name in r2_series.index]
r2_series.plot(kind='barh', ax=axes[2], color=bar_colors, edgecolor='white')
axes[2].set_title('All Models R2 -- Including Tuned', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Test R2 Score')
for i, (name, val) in enumerate(r2_series.items()):
    axes[2].text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Hyperparameter Tuning Impact', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nTuning Summary:')
for model_name in ['Random Forest', 'XGBoost']:
    tc = tuning_comparison[model_name]
    r2_change = tc['After R2'] - tc['Before R2']
    print(f'  {model_name}:')
    print(f'    R2:   {tc["Before R2"]:.4f} -> {tc["After R2"]:.4f} ({"+" if r2_change >= 0 else ""}{r2_change:.4f})')
    print(f'    MAE:  {tc["Before MAE"]:.4f} -> {tc["After MAE"]:.4f}')

---
## Limitations & Future Work

### Limitations
1. **Synthetic Data** -- While realistic, synthetic data may not capture all real-world complexities
2. **No Temporal Features** -- Time-series effects (trend, seasonality) are not modeled
3. **Feature Interactions** -- Complex multi-way interactions may be missed by simpler models
4. **Static Environment** -- Real factories have dynamic conditions (machine degradation, workforce changes)

### Future Improvements
1. Integrate real factory production data
2. Add time-series models (ARIMA, LSTM) for trend-aware predictions
3. Engineer lag features (previous batch times, rolling averages)
4. Implement SHAP values for model interpretability
5. Real-time prediction pipeline with live sensor data

In [ ]:
# ============================================================
# CELL 15: Save Best Model & Artifacts
# ============================================================
import os

joblib.dump(final_best_model, 'production_time_model.pkl')
print(f'[OK] Saved best model ({final_best_name}): production_time_model.pkl')

joblib.dump(scaler, 'scaler.pkl')
print('[OK] Saved StandardScaler: scaler.pkl')

# Save all label encoders
encoders = {
    'le_product': le_product,
    'le_machine': le_machine,
    'le_shift': le_shift,
    'le_day': le_day
}
joblib.dump(encoders, 'label_encoders.pkl')
print('[OK] Saved LabelEncoders: label_encoders.pkl')

feature_names = list(X.columns)
joblib.dump(feature_names, 'feature_names.pkl')
print(f'[OK] Saved feature names ({len(feature_names)} features): feature_names.pkl')

comparison_df.to_csv('model_comparison.csv', index=False)
print('[OK] Saved model comparison: model_comparison.csv')

print(f'\n{"="*60}')
print(f'All artifacts saved to: {os.getcwd()}')
print(f'{"="*60}')

# List saved files
for f in sorted(os.listdir('.')):
    if f.endswith(('.pkl', '.csv', '.py', '.txt')) and not f.startswith('.'):
        size = os.path.getsize(f) / 1024
        print(f'   {f:.<40} {size:.1f} KB')

---
## Phase 7: Streamlit Application

The Streamlit app (`app.py`) is in the project directory.

**Features:**
- **Predict Tab**: Interactive inputs for all production parameters
- **Model Insights Tab**: Feature importance, model comparison charts
- **Data Explorer Tab**: Upload and explore datasets

**Deployment:**
- Push to GitHub with all `.pkl` files and `app.py`
- Deploy on [Streamlit Community Cloud](https://share.streamlit.io/)
- Or run locally: `streamlit run app.py`

In [ ]:
# ============================================================
# CELL 16: Generate requirements.txt
# ============================================================

requirements = (
    "pandas>=2.0.0\n"
    "numpy>=1.24.0\n"
    "scikit-learn>=1.3.0\n"
    "xgboost>=2.0.0\n"
    "matplotlib>=3.7.0\n"
    "seaborn>=0.12.0\n"
    "scipy>=1.11.0\n"
    "joblib>=1.3.0\n"
    "streamlit>=1.28.0\n"
    "plotly>=5.17.0\n"
)

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('[OK] requirements.txt generated')
print('\nInstall: pip install -r requirements.txt')

print('\nFinal Project Structure:')
import os
for f in sorted(os.listdir('.')):
    if not f.startswith('.') and not f.startswith('__'):
        icon = '[DIR]' if os.path.isdir(f) else '[FILE]'
        size = ''
        if os.path.isfile(f):
            s = os.path.getsize(f) / 1024
            size = f'({s:.1f} KB)'
        print(f'   {icon} {f} {size}')